## 1.3: Snowflake Schema


In [0]:
%sql
-- Nivel 1: Provincia
DROP TABLE IF EXISTS propiedades.gold.dim_provincia_sf;

CREATE TABLE propiedades.gold.dim_provincia_sf 
(
    provincia_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    provincia STRING NOT NULL,
    pais STRING DEFAULT 'Argentina',
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de provincias - Snowflake Schema';



INSERT INTO propiedades.gold.dim_provincia_sf (provincia, pais)
SELECT DISTINCT 'Buenos Aires' as provincia, 'Argentina' as pais;


In [0]:
%sql

-- Nivel 2: Ciudad
DROP TABLE IF EXISTS propiedades.gold.dim_ciudad_sf;

CREATE TABLE propiedades.gold.dim_ciudad_sf (
    ciudad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    ciudad STRING NOT NULL,
    provincia_id BIGINT,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de ciudades - Snowflake Schema';


INSERT INTO propiedades.gold.dim_ciudad_sf (ciudad, provincia_id)
SELECT DISTINCT
    CASE 
        WHEN sp.region = 'capital federal' THEN 'CABA'
        ELSE 'GBA'
    END as ciudad,
    1 as provincia_id
FROM propiedades.silver.propiedades sp
WHERE sp.region IS NOT NULL;



In [0]:
%sql
-- Nivel 3: Zona (referencia a ciudad)
DROP TABLE IF EXISTS propiedades.gold.dim_zona_sf;

CREATE TABLE propiedades.gold.dim_zona_sf (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    partido STRING NOT NULL,
    region STRING NOT NULL,
    ciudad_id BIGINT,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de zonas - Snowflake Schema (normalizada)';

INSERT INTO propiedades.gold.dim_zona_sf 
(partido, region, ciudad_id)

SELECT DISTINCT
        sp.partido,
        sp.region,
        dc.ciudad_id

FROM propiedades.silver.propiedades sp
JOIN propiedades.gold.dim_ciudad_sf dc ON 
    CASE 
        WHEN sp.region = 'capital federal' THEN 'CABA'
        ELSE 'GBA'
    END = dc.ciudad
WHERE sp.partido IS NOT NULL
ORDER BY sp.partido;


In [0]:
%sql

-----------------------------------------------------------------------
-- Query de ejemplo Snowflake
-----------------------------------------------------------------------
SELECT  
        dp.provincia,
        dc.ciudad,
        dz.partido, dz.region
        
FROM propiedades.silver.propiedades sp
JOIN propiedades.gold.dim_zona_sf dz      ON sp.partido = dz.partido AND sp.region = dz.region
JOIN propiedades.gold.dim_ciudad_sf dc    ON dz.ciudad_id = dc.ciudad_id
JOIN propiedades.gold.dim_provincia_sf dp ON dc.provincia_id = dp.provincia_id
LIMIT 20;